# ADB Indonesia — Targeted RRP/PCR recovery (89 gaps)

This notebook **does not use** the broken `projects/documents?searchstax...` endpoint. It checks a limited set of official ADB PDF URL patterns for the **89 unresolved project-document combinations**, saves only verified PDFs, and exports a ZIP plus an audit. It **cannot guarantee** that every report exists or will be found. In particular, policy-based program PCRs may use multi-project filenames; those require review.

**Instructions:** Open in Google Colab and run cells in order. No uploads are required: the gap list is embedded. Do not mark `NOT_FOUND` as evidence of nonpublication. It means only that the URL patterns tested did not yield a valid PDF. The output ZIP contains *only new recovery downloads*, not the previous 103 PDFs.

In [ ]:
!pip -q install requests pandas

In [ ]:
from pathlib import Path
import pandas as pd, requests, hashlib, time, zipfile, io, re
from urllib.parse import urlparse
from google.colab import files

ROOT=Path('/content/ADB_Targeted_Recovery'); ROOT.mkdir(exist_ok=True)
OUT=ROOT/'Recovered_PDFs'; OUT.mkdir(exist_ok=True)
GAPS=pd.DataFrame([{'project_id': '38354-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'State Accountability Revitalization Project'}, {'project_id': '50297-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Project'}, {'project_id': '50395-006', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Growth Project'}, {'project_id': '51114-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': '(Phase 2)'}, {'project_id': '51157-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Flood Management in North Java Project'}, {'project_id': '52282-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Geothermal Power Generation Project'}, {'project_id': '52316-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Reconstruction'}, {'project_id': '52332-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Innovation Project'}, {'project_id': '53211-003', 'missing_type': 'PCR', 'project_title_from_prior_audit': '3'}, {'project_id': '53220-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Program - Western and Central Java'}, {'project_id': '54152-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Green Finance Facility (Phase 1)'}, {'project_id': '54224-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Transformation Program'}, {'project_id': '54224-002', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Project'}, {'project_id': '54256-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Sector Project'}, {'project_id': '54428-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Citywide Inclusive Sanitation Project'}, {'project_id': '55020-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Aquaculture Project'}, {'project_id': '55063-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Parks Project'}, {'project_id': '56257-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Subprogram 1'}, {'project_id': '57074-001', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Markets - Phase 2 Preparation Project'}, {'project_id': '57074-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Markets - Phase 2 Preparation Project'}, {'project_id': '52282-003', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Additional Financing)'}, {'project_id': '54461-003', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Development Program (Subprogram 3)'}, {'project_id': '56017-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Trans South-South Java Road Project'}, {'project_id': '56140-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Transition Program (Phase 1)'}, {'project_id': '32507-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'and Social Security Reform Program II)'}, {'project_id': '32507-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'and Social Security Reform Program II)'}, {'project_id': '32507-023', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Subprogram 2'}, {'project_id': '32507-023', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Subprogram 2'}, {'project_id': '33409-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Vocational Education Strengthening Project'}, {'project_id': '34100-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Renewable Energy Development'}, {'project_id': '34152-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Community Water Services and Health'}, {'project_id': '34152-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Community Water Services and Health'}, {'project_id': '34159-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Road Rehabilitation II'}, {'project_id': '34159-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Road Rehabilitation II'}, {'project_id': '35139-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Power Transmission Improvement Sector'}, {'project_id': '35182-043', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Sector Project'}, {'project_id': '35183-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Food Security and Poverty Reduction'}, {'project_id': '36541-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Reform Program'}, {'project_id': '36541-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Reform Program'}, {'project_id': '36541-023', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Reform (Project Loan)'}, {'project_id': '36541-023', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Reform (Project Loan)'}, {'project_id': '37049-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Investment Program'}, {'project_id': '37049-023', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Management Investment Program - Project 1'}, {'project_id': '37475-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Madrasah Education Development'}, {'project_id': '37475-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Madrasah Education Development'}, {'project_id': '38117-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Empowerment'}, {'project_id': '38264-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Governance Reform Program - Subprogram 1'}, {'project_id': '38385-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Mandiri Project II'}, {'project_id': '38479-034', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Regional Roads Development Project'}, {'project_id': '39071-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Health Project'}, {'project_id': '39597-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Rural Infrastructure Support Project'}, {'project_id': '39597-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Rural Infrastructure Support Project'}, {'project_id': '40003-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Development Goals Acceleration Program'}, {'project_id': '40009-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Infrastructure Project Development Facility'}, {'project_id': '40009-023', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Program (Subprogram 1)'}, {'project_id': '40009-033', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Program (Subprogram 2)'}, {'project_id': '40009-043', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Program - Subprogram 3'}, {'project_id': '40061-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Improvement Project'}, {'project_id': '40247-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Project'}, {'project_id': '40247-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Project'}, {'project_id': '40272-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Third Development Policy Support Program'}, {'project_id': '40272-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Third Development Policy Support Program'}, {'project_id': '40273-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Fourth Development Policy Support Program'}, {'project_id': '40273-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Fourth Development Policy Support Program'}, {'project_id': '42099-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Polytechnics Education Development Project'}, {'project_id': '42109-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Company Project'}, {'project_id': '43009-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Public Expenditure Support Facility Program'}, {'project_id': '43009-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Public Expenditure Support Facility Program'}, {'project_id': '43092-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Fifth Development Policy Support Program'}, {'project_id': '43092-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Fifth Development Policy Support Program'}, {'project_id': '43211-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Sixth Development Policy Support Program'}, {'project_id': '43211-013', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Sixth Development Policy Support Program'}, {'project_id': '43251-025', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Investment Project'}, {'project_id': '43251-025', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Investment Project'}, {'project_id': '43255-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Support to PNPM Mandiri Project'}, {'project_id': '44252-013', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Program'}, {'project_id': '46093-004', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Connectivity Program, Subprogram 2'}, {'project_id': '46239-001', 'missing_type': 'RRP', 'project_title_from_prior_audit': 'Precautionary Financing Facility'}, {'project_id': '49043-002', 'missing_type': 'RRP', 'project_title_from_prior_audit': '(Subprogram 2)'}, {'project_id': '52218-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Program (Subprogram 1)'}, {'project_id': '52218-002', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Program (Subprogram 2)'}, {'project_id': '53211-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': '1'}, {'project_id': '53211-002', 'missing_type': 'PCR', 'project_title_from_prior_audit': '2'}, {'project_id': '54461-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Development Program (Subprogram 1)'}, {'project_id': '54461-002', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Development Program, Subprogram 2'}, {'project_id': '55235-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Subprogram 1'}, {'project_id': '56368-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Program, Subprogram 1'}, {'project_id': '57018-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': '1'}, {'project_id': '58165-001', 'missing_type': 'PCR', 'project_title_from_prior_audit': 'Program, Subprogram 3'}])
print('Gaps:',len(GAPS),'| Projects:',GAPS.project_id.nunique())
print(GAPS.missing_type.value_counts())


## Probe verified PDF paths

ADB filenames vary. This probes only two standard single-project PDF patterns per record, without guessing report availability. Requests are rate-limited; HTTP 403/429 stops the run rather than circumventing restrictions. Existing files are reused when rerun. Checkpoints are saved after each row.

In [ ]:
session=requests.Session()
session.headers.update({'User-Agent':'Mozilla/5.0 (compatible; academic document retrieval)'})
AUDIT=ROOT/'recovery_audit.csv'
existing=pd.read_csv(AUDIT).fillna('') if AUDIT.exists() else pd.DataFrame()
completed=set(zip(existing.project_id.astype(str),existing.document_type.astype(str))) if len(existing) else set()
rows=existing.to_dict('records') if len(existing) else []

for k, r in GAPS.iterrows():
    pid=str(r.project_id); kind=str(r.missing_type).lower()
    if (pid,kind.upper()) in completed:
        continue
    parent=pid.split('-')[0]
    filenames=[f'{pid}-{kind}-en.pdf',f'{pid}-{kind}.pdf']
    urls=[f'https://www.adb.org/sites/default/files/project-documents/{parent}/{name}' for name in filenames]
    result={'project_id':pid,'document_type':kind.upper(),'project_title':r.project_title_from_prior_audit,
            'status':'NOT_FOUND_BY_PATTERN','pdf_url':'','local_file':'','sha256':'','attempted_urls':' | '.join(urls),'error':''}
    for url in urls:
        try:
            response=session.get(url,timeout=45,stream=True,allow_redirects=True)
            if response.status_code in (403,429):
                result['status']='ACCESS_RESTRICTED';result['error']=f'HTTP {response.status_code}'
                response.close();break
            if response.status_code != 200:
                result['error']=f'Last HTTP {response.status_code}';response.close();continue
            data=bytearray()
            for chunk in response.iter_content(65536):
                if chunk: data.extend(chunk)
                if len(data)>30*1024*1024: raise ValueError('Over 30MB safety limit')
            response.close()
            if not bytes(data[:5])==b'%PDF-':
                result['error']='HTTP 200 but not PDF';continue
            dest=OUT/pid;dest.mkdir(exist_ok=True)
            target=dest/f'{pid}_{kind.upper()}_{urlparse(url).path.split("/")[-1]}'
            target.write_bytes(data)
            result.update(status='DOWNLOADED',pdf_url=url,local_file=str(target.relative_to(ROOT)),
                          sha256=hashlib.sha256(data).hexdigest(),error='')
            break
        except Exception as exc:
            result['error']=str(exc)[:240]
        time.sleep(1)
    rows.append(result)
    pd.DataFrame(rows).to_csv(AUDIT,index=False)
    print(f'[{len(rows)}/{len(GAPS)}] {pid} {kind.upper()}: {result["status"]}',flush=True)
    if result['status']=='ACCESS_RESTRICTED':
        print('Stopped due to access restriction; rerun later only if authorized.');break
    time.sleep(2)

report=pd.DataFrame(rows)
print('\nSUMMARY\n',report.status.value_counts().to_string())
print('New PDFs:',len(list(OUT.rglob('*.pdf'))))


## Export the new PDFs and full audit

The ZIP contains any successfully recovered PDFs, `recovery_audit.csv`, and a separate list of unresolved cases. **Do not assume the unresolved reports are unpublished.**

In [ ]:
report=pd.read_csv(ROOT/'recovery_audit.csv').fillna('')
report[report.status!='DOWNLOADED'].to_csv(ROOT/'still_unresolved.csv',index=False)
archive=ROOT/'ADB_Indonesia_Targeted_Recovery.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*.pdf'):z.write(p,p.relative_to(ROOT))
    for name in ['recovery_audit.csv','still_unresolved.csv']:
        z.write(ROOT/name,name)
print('ZIP:',archive,'|',round(archive.stat().st_size/1024/1024,2),'MB')
files.download(str(archive))


## After running

Upload `recovery_audit.csv` here (from the ZIP). We can then check newly recovered reports, and use project-title/loan-number searches for cases still unresolved. This limited recovery method intentionally does **not** claim to search the entire ADB document catalogue.